# GCSI Topic Hierarchy

## What moved the reading? Topic → driver → sector

GCSI topics form a three-level tree: **sector** → **L1 driver** (Market, Macro, Supply, Demand, Corporate) →
**L2 topic**. This notebook uses it to:

1. Roll a commodity's topics up to its drivers, and compare drivers over time
2. **Attribute** a change in the overall reading exactly to the topics that caused it
3. Compare commodities in the same sector, driver by driver

Input is the daily panel written by `gcsi_bulk_history.ipynb` (`data/gcsi_daily/<TICKER>_daily.parquet`). Run
that first.

> **Disclaimer:** This notebook is provided for informational and research purposes only. Nothing here constitutes financial advice or a recommendation to buy, sell, or hold any asset. Sentiment scores are model outputs describing how news was judged at publication; they are not price forecasts.

In [ ]:
%pip install pandas numpy matplotlib seaborn pyarrow

## Step 1: Configuration and data

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

TOPIC_SEPARATOR = '::'  # Sector::L1::L2, the form the API returns


def normalise_topic(labels: pd.Series) -> pd.Series:
    """Write each label as sector, driver and topic joined by TOPIC_SEPARATOR, whatever separator the source used."""
    return labels.astype('string').str.replace(r'\s*(?:::|-)\s*', TOPIC_SEPARATOR, regex=True)


def load_taxonomy() -> pd.DataFrame:
    """The topic hierarchy in reference/gcsi_topics.csv, found by walking up from the working directory."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        csv = base / 'reference' / 'gcsi_topics.csv'
        if csv.exists():
            return pd.read_csv(csv)
    raise FileNotFoundError('reference/gcsi_topics.csv not found: run this notebook from inside the repository')


def add_hierarchy(frame: pd.DataFrame, taxonomy: pd.DataFrame) -> pd.DataFrame:
    """Attach sector, l1 (driver) and l2 to each row, whatever separator the topic labels use."""
    lookup = {TOPIC_SEPARATOR.join([row.sector, row.l1, row.l2]): (row.sector, row.l1, row.l2)
              for row in taxonomy.itertuples()}
    parts = normalise_topic(frame['topic_name']).map(lookup)
    unknown = frame.loc[parts.isna(), 'topic_name'].unique()
    if len(unknown):
        print(f'{len(unknown)} topic label(s) not in the reference file, e.g. {unknown[:3].tolist()}')
    frame = frame.copy()
    frame[['sector', 'l1', 'l2']] = pd.DataFrame(parts.tolist(), index=frame.index, columns=['sector', 'l1', 'l2'])
    frame['driver'] = frame['sector'] + TOPIC_SEPARATOR + frame['l1']
    return frame


PANEL_DIR = Path('data/gcsi_daily')
TICKER = 'BZ_COM'
COMPARE_TICKERS = ['BZ_COM', 'CL_COM']   # commodities in the same sector
INDEX_TYPE = 'COMBINED'
SENTIMENT_TYPE = 'short_term'
RECENT_DAYS, PRIOR_DAYS = 7, 28          # attribution: last 7 days against the 28 days before
MIN_LINKS = 10                           # leave out cells with fewer topic-links than this

taxonomy = load_taxonomy()


def load_topics(ticker: str) -> pd.DataFrame:
    panel = pd.read_parquet(PANEL_DIR / f'{ticker}_daily.parquet')
    panel = panel[(panel['level'] == 'topic') & (panel['index_type'] == INDEX_TYPE)
                  & (panel['sentiment_type'] == SENTIMENT_TYPE)].rename(columns={'name': 'topic_name'})
    return add_hierarchy(panel, taxonomy)


topics = load_topics(TICKER)
end = topics['date'].max()
print(f"{TICKER}: {topics['topic_name'].nunique()} topics in {topics['sector'].nunique()} sector(s), "
      f"{topics['date'].min():%Y-%m-%d} → {end:%Y-%m-%d}")

## Step 2: Drivers over the last year

Monthly, topic-weighted average by driver. A driver's average is `Σ sentiment_sum / Σ headline_count` over its
topics, so busy topics weigh more, as they should.

In [ ]:
year = topics[topics['date'] > end - pd.Timedelta(days=365)]
monthly = (
    year.assign(month=year['date'].dt.to_period('M').dt.to_timestamp())
    .groupby(['l1', 'month'])[['sentiment_sum', 'headline_count']].sum()
)
drivers = (monthly['sentiment_sum'] / monthly['headline_count']).where(monthly['headline_count'] >= MIN_LINKS).unstack()
fig, ax = plt.subplots(figsize=(14, 3.5))
sns.heatmap(drivers, cmap='RdYlGn', center=0, annot=True, fmt='.2f', linewidths=0.4, ax=ax,
            xticklabels=[m.strftime('%b %y') for m in drivers.columns], cbar_kws={'label': 'Average sentiment'})
ax.set_title(f'{TICKER} | {SENTIMENT_TYPE} | monthly average by driver')
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

share = monthly['headline_count'].unstack(0)
(share.div(share.sum(axis=1), axis=0)).plot.area(figsize=(14, 3.5), alpha=0.8, title='Share of topic-links by driver')
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()

## Step 3: Exact attribution of a change

The overall reading is a weighted average: `A = Σᵢ wᵢ·aᵢ`, where `wᵢ` is topic i's share of topic-links and `aᵢ`
is its average. The change between two windows splits exactly into per-topic terms:

```
ΔA = Σᵢ (w₁ᵢ·a₁ᵢ − w₀ᵢ·a₀ᵢ)
```

Each term combines a **tone** effect (the topic's average changed) and a **mix** effect (the topic got more or
less of the news). The terms add up to the headline change, so nothing is left unexplained.

In [ ]:
def window(frame: pd.DataFrame, start: pd.Timestamp, stop: pd.Timestamp) -> pd.DataFrame:
    sub = frame[(frame['date'] > start) & (frame['date'] <= stop)]
    out = sub.groupby(['topic_name'])[['sentiment_sum', 'headline_count']].sum()
    out['weight'] = out['headline_count'] / out['headline_count'].sum()
    out['avg'] = out['sentiment_sum'] / out['headline_count']
    return out


recent = window(topics, end - pd.Timedelta(days=RECENT_DAYS), end)
prior = window(topics, end - pd.Timedelta(days=RECENT_DAYS + PRIOR_DAYS), end - pd.Timedelta(days=RECENT_DAYS))
both = recent.merge(prior, how='outer', left_index=True, right_index=True, suffixes=('_recent', '_prior'))
both = both.fillna({'weight_recent': 0, 'weight_prior': 0})
both['contribution'] = (both['weight_recent'] * both['avg_recent'].fillna(0)
                        - both['weight_prior'] * both['avg_prior'].fillna(0))
a_recent = (recent['weight'] * recent['avg']).sum()
a_prior = (prior['weight'] * prior['avg']).sum()
print(f'Overall: {a_prior:+.3f} → {a_recent:+.3f}  (change {a_recent - a_prior:+.3f}; '
      f"sum of topic contributions {both['contribution'].sum():+.3f})")

top = both.reindex(both['contribution'].abs().sort_values(ascending=False).index).head(10).reset_index()
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.barh(top['topic_name'][::-1], top['contribution'][::-1],
        color=np.where(top['contribution'][::-1] > 0, '#27ae60', '#c0392b'))
ax.axvline(0, color='grey', lw=0.8)
ax.set_title(f'{TICKER}: what moved the {SENTIMENT_TYPE} reading (last {RECENT_DAYS}d vs prior {PRIOR_DAYS}d)')
plt.tight_layout()
plt.show()
display(top[['topic_name', 'headline_count_prior', 'headline_count_recent', 'avg_prior', 'avg_recent',
             'contribution']].round(3))

## Step 4: Across a sector

The same driver for different commodities. Brent and WTI share much of their news, so they often move together.
Where they diverge is usually the interesting part.

In [ ]:
rows = []
for ticker in COMPARE_TICKERS:
    path = PANEL_DIR / f'{ticker}_daily.parquet'
    if not path.exists():
        print(f'{ticker}: no daily panel (run gcsi_bulk_history.ipynb for it)')
        continue
    sub = load_topics(ticker)
    sub = sub[sub['date'] > end - pd.Timedelta(days=90)]
    grouped = sub.groupby('l1')[['sentiment_sum', 'headline_count']].sum()
    rows.append((grouped['sentiment_sum'] / grouped['headline_count']).where(grouped['headline_count'] >= MIN_LINKS).rename(ticker))

if rows:
    grid = pd.concat(rows, axis=1).T
    fig, ax = plt.subplots(figsize=(9, 0.6 * len(grid) + 1.5))
    sns.heatmap(grid, cmap='RdYlGn', center=0, annot=True, fmt='.3f', linewidths=0.4, ax=ax)
    ax.set_title(f'Last 90 days | {SENTIMENT_TYPE} | average by driver')
    plt.tight_layout()
    plt.show()